In [ ]:
import pandas as pd

df = pd.read_csv('amazon_products_sales_data_uncleaned.csv')
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
for col in ['rating', 'is_best_seller', 'is_sponsored', 'is_couponed']:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

## Data Investigation Summary

### Dataset Overview
- amazon_products_sales_data, 42675, 16, a dataset representing a product listings data, not sales records

### Missing Values
- rating                     1024  legitimate null
number_of_reviews            1024   legitimate null 
bought_in_last_month         3217    legitimate null 
current/discounted_price    11749    legitimate null
buy_box_availability        14653    legitimate null
delivery_details            11720    true error
sustainability_badges       39267    legitimate null
product_url                  2069,   true error

### Data Type Issues
- rating,bought_in_last_month, price_on_variant,listed_price,number_of_reviews,current/discounted_price,collected_at

### Column-Level Issues
- rating: stored as text sentence
- is_best_seller: contains mixed/corrupted values
- price_on_variant: contains non-price data
- current/discounted_price: null when no discount exists (legitimate)
- buy_box_availability: single-value column, low analytical use

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean['rating'] = df_clean['rating'].str.split(" ").str[0]
df_clean['rating'] = pd.to_numeric(df_clean['rating'], errors='coerce')

In [ ]:
df_clean['rating'].dtype

In [ ]:
print(df_clean['rating'])

## Cleaning Log

### rating
- Issue: stored as text sentence e.g. "4.6 out of 5 stars"
- Fix: extracted numeric value using str.split(), converted to float64

In [ ]:
df_clean['number_of_reviews'].value_counts().head(10)

In [ ]:
df_clean['number_of_reviews'].dtype

In [ ]:
df_clean['number_of_reviews'].str.contains(',').sum()

In [ ]:
df_clean['number_of_reviews'] = pd.to_numeric(df_clean['number_of_reviews'], errors='coerce').astype('Int64')

In [ ]:
df_clean['number_of_reviews'].dtype

In [ ]:
print(df_clean['number_of_reviews'])

### number_of_reviews
- Issue: comma-formatted numbers stored as text e.g. "35,882"
- Fix: removed commas with str.replace(), converted to Int64

In [ ]:
df_clean['bought_in_last_month'].value_counts().head(10)

In [ ]:
def clean_bought(value):
    if pd.isna(value):
        return None
    
    first = value.split(" ")[0]
    first = first.replace('+', '')
    
    if 'K' in first:
        return int(first.replace('K','')) * 1000
    else:
        try:
            return int(first)
        except:
            return None

In [ ]:
df_clean['bought_in_last_month'] = df_clean['bought_in_last_month'].apply(clean_bought)
df_clean['bought_in_last_month'].dtype

In [ ]:
df_clean['bought_in_last_month'] = pd.to_numeric(df_clean['bought_in_last_month'], errors='coerce').astype('Int64')
df_clean['bought_in_last_month'].dtype

### bought_in_last_month
- Issue: text format e.g. "100+ bought in past month", "1K+ bought in past month", non-purchase labels mixed in
- Fix: wrote custom function to extract numeric value, convert K to thousands, handle invalid entries, converted to Int64

In [ ]:
df_clean['listed_price'].value_counts().head(10)

In [ ]:
df_clean['listed_price'].dtype


In [ ]:
df_clean['listed_price'] = df_clean['listed_price'].str.replace('$', '', regex=False)
df_clean['listed_price'] = pd.to_numeric(df_clean['listed_price'], errors='coerce')
df_clean['listed_price'].dtype

### listed_price
- Issue: $ symbol stored as text, "No Discount" for full price products
- Fix: removed $ with regex=False, converted to float64, "No Discount" became NaN via errors='coerce'

In [ ]:
df_clean['current/discounted_price'].value_counts().head(10)

In [ ]:
df_clean['current/discounted_price'].dtype
df_clean['current/discounted_price'].str.contains('\$', regex=True).sum()

In [ ]:
df_clean['current/discounted_price'].dtype

In [ ]:
df_clean['current/discounted_price'] = df_clean['current/discounted_price'].str.replace('$', '', regex=False)
df_clean['current/discounted_price'] = pd.to_numeric(df_clean['current/discounted_price'], errors='coerce')
df_clean['current/discounted_price'].dtype


### current/discounted_price
- Issue: stored as text, $ symbol present in some values
- Fix: removed $ with regex=False, converted to float64

In [ ]:
df_clean['collected_at'] = pd.to_datetime(df_clean['collected_at'])
df_clean['collected_at'].dtype

### collected_at
- Issue: date stored as text
- Fix: converted to datetime using pd.to_datetime()

In [ ]:
df_clean['price_on_variant'].value_counts().head(10)

In [ ]:
df_clean['price_on_variant']=df_clean['price_on_variant'].str.replace("basic variant price: ",'')
df_clean['price_on_variant']= df_clean['price_on_variant'].str.replace('$', '', regex=False)
df_clean['price_on_variant']=pd.to_numeric(df_clean['price_on_variant'], errors='coerce')

In [ ]:
df_clean['price_on_variant'].dtype

In [ ]:
print(df_clean['price_on_variant'].value_counts().head(10))

### price_on_variant
- Issue: values prefixed with "basic variant price: ", $ symbol present, non-price data mixed in e.g. "16 Count (Pack of 1)"
- Fix: removed prefix with str.replace(), removed $ with regex=False, converted to float64, invalid entries became NaN via errors='coerce'
- As you can see in the last code ran that the 'price_on_variant' has now becom eclean, usable values that are ready to be used for analysis

In [ ]:
df_clean['is_best_seller'].value_counts()

In [ ]:
valid_values = ['No Badge', 'Best Seller', "Amazon's"]
df_clean['is_best_seller'] = df_clean['is_best_seller'].where(
    df_clean['is_best_seller'].isin(valid_values), other=None
)

In [ ]:
df_clean['is_best_seller'].value_counts()

### is_best_seller
- Issue: promotional labels mixed in e.g. "Save 30%", "Limited time deal", "Ends in" that are not valid entries for the 'is_best_seller' column.
- Fix: defined valid values (list), replaced invalid entries with None using .where() and .isin(), .isin() runs through the code and the value thaat meet its condition is returns True. While .where() then performs a function for all true outcomes and returns None for all Fallse outcomes. This ensure only valid entries remain in the 'is_best_seller' column.

In [ ]:
df_clean['rating'].mean()
df_clean['rating'] = df_clean['rating'].fillna(df_clean['rating'].mean())
df_clean.dropna(subset=['product_url'], inplace=True)

In [ ]:
#now we can save the cleaned data to a new CSV file and we will run isnull() to see the new value of missing value in the cleaned data
df_clean.isnull().sum()

### Missing Values
- rating: filled with column mean (honest representation of unknown ratings)
- product_url: dropped rows with missing URLs (unusable product records)
- All other nulls: left as legitimate NaN (not applicable entries). This reveals that cleaning reveals the true extent of missing data. The original null count was an undercount because bad data was hiding as text.

In [ ]:
df_clean.shape

In [ ]:
df_clean.head(5)

### Phase 4 — Business Logic and Patterns
- With the data now cleaned here is an example of what you can now do with the data insights, that would matter to an Amazon seller or a business analyst.
1. What rating profile do best selling products have?
2. Does offering a discount increase sales?
3. Does offering a coupon increase sales?

In [ ]:

# We will group the data by the 'is_best_seller' column and calculate the mean rating for each group. The result will be rounded to 2 decimal places for better readability.
df_clean.groupby('is_best_seller')['rating'].mean().round(2)

###
Best Seller products average 4.53 stars. No Badge products average 4.38 stars. That's a consistent gap.
Best selling products tend to have higher ratings than products with no badge. Customers are buying and endorsing higher quality products more.
If you're a seller, this tells you that, rating matters. Products with higher ratings are more likely to become best sellers.

In [ ]:
#does offering a discount increase sales?
df_clean['has_discount'] = df_clean['current/discounted_price'].notna()
df_clean.groupby('has_discount')['bought_in_last_month'].mean().round(0)

### products with a discount sell more. 512 vs 405 bought per month.

In [ ]:
#For Coupons
df_clean['has_coupon'] = df_clean['is_couponed'] != 'No Coupon'
df_clean.groupby('has_coupon')['bought_in_last_month'].mean().round(0)

Products with coupons sell even more than products with discounts. 739 vs 478 bought per month. Coupons have a bigger impact on sales than discounts.
Here are three business findings:

Higher rated products are more likely to be best sellers
Discounted products sell more (512 vs 405)
Products with coupons sell the most (739 vs 478)

In [ ]:

df_clean.head()